# 2.2 怎麼同時看前面幾個字？


看到「我想喝」和「我想看」，即使最後一字不同，前面兩個字也能提供線索。上一章每次只查目前一字，現在要讓一個問題同時帶入前三個位置。[2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html) 已經把每個字查成特徵向量；最直接的做法是不先混合，而是按順序把三排數字接成一長排。這叫拼接（concatenation），像把三張有固定欄位的小卡貼在同一張長卡上。

用每字兩個特徵的玩具表，ID1對應 `[1,10]`，ID2是 `[2,20]`，ID3是 `[3,30]`。第一筆前文ID順序是1、2、3，拼成 `[1,10,2,20,3,30]`；第二筆3、2、1則拼成 `[3,30,2,20,1,10]`。沒有把數字相加，也沒有丟掉順序。後面配方層的不同輸入位置會使用不同權重，所以它可以知道哪兩格來自最前面的字。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from torch import nn

embedding = nn.Embedding(4, 2)
with torch.no_grad():
    embedding.weight.copy_(torch.tensor([[0.0, 0.0], [1.0, 10.0], [2.0, 20.0], [3.0, 30.0]]))
ids = torch.tensor([[1, 2, 3], [3, 2, 1]])
x = embedding(ids)
context = x.reshape(2, -1)
print(x.shape)
print(context)
assert context.shape == (2, 6)

`with torch.no_grad():` 包住手動設定表格的部分，讓 `copy_` 把指定數字抄入可調表格時，不將這次原地設定記錄成求導運算；這與[1.12的參數更新範圍](https://birdhackor.github.io/tiny-perceptron-vlm/1.12.html)同樣用來允許直接修改參數。離開縮排後，查表和拼接會照常記錄求導關係。

輸入 `ids` 形狀 `[2,3]`，表示兩筆、每筆三位置。查表後是 `[2,3,2]`，每位置多了一排兩個特徵。`reshape(2,-1)` 保留兩筆，把每筆剩下的格子依原順序排成一軸；這裡的-1不是最後一軸索引，而是要求PyTorch由總格數推算該軸長度。總共有12個數，分兩筆，因而自動算出每筆6格。輸出正是前段兩條長卡。列印的 `grad_fn=<ViewBackward0>` 是PyTorch附上的求導關係名稱，記住 `context` 如何由查表結果重新分組，日後求梯度可以沿這條關係傳回表格；此處尚未呼叫 `backward()` 或更新參數。

把形狀寫成 `[B,C,D] → [B,C×D]`，B是同時有幾筆例子，C是每筆能看的位置數，D是每位置特徵數。三個字母不是特定數值，更不是PyTorch指令；這次B=2、C=3、D=2。reshape只改分組，不創造特徵，軸的背景可回看 [W.3](https://birdhackor.github.io/tiny-perceptron-vlm/first-steps.html#W.3)。

這個做法的限制也是它的優點：每個位置有固定欄位，容易追蹤，但前文長度C要先決定。若一會給三字、一會給五字，長卡格數不同，後面的配方層無法直接共用同一個輸入大小。可以先規定固定窗口，太長只取最近C個位置，太短則補齊；補齊策略會在第7章仔細講。這裡用兩筆恰好三字，不把新問題混進例子。

練習只把第一筆改成 `[2,1,3]`，先寫出預期長卡 `[2,20,1,10,3,30]`，再跑核對。第二筆應原封不動，形狀仍 `[2,6]`。如果你只看總和，兩種順序可能一樣；直接看分段位置，才知道拼接保留了哪些資訊。
